# 0.1 Overview

The representative population pool is generated independently from multiple demographic datasets.

Although each attribute is statistically plausible, some attribute combinations are logically inconsistent.

This notebook applies a series of logical constraints to improve internal consistency before constructing behaviour profiles.

Input dataset:

- Representative Population Pool

Output:

- Representative Population Pool with Constraint Handlings

### 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

### 2. Set Up File Paths

In [2]:
from pathlib import Path

project_root = Path.cwd().parents[1]

input_path = (
    project_root
    / "output"
    / "demographic_profiles"
)

output_path = (
    project_root
    / "output"
    / "demographic_profiles"
)

output_path.mkdir(
    parents=True,
    exist_ok=True
)

### 3. Load Dataset

In [3]:
population_pool_file = (
    input_path
    / "representative_population_pool.parquet"
)

population_pool = pd.read_parquet(
    population_pool_file
)

display(population_pool.head())

print(population_pool.shape)

,Age,Gender,Household Type,Household Count,Economic Activity Status,Economic Count,Industry,Industry Count,Ethnic Group,Ethnic Count,Distance travelled to work,Travel Distance Count,Car Ownership,Car Ownership Count
0,0-4,Female,Couple household: All children non-dependent,0,Does not apply,34378,"A Agriculture, forestry and fishing",0,"Asian, Asian British or Asian Welsh: Bangladeshi",833,10km to less than 30km,0,1 car or van in household,3139
1,0-4,Female,Couple household: All children non-dependent,0,Does not apply,34378,"A Agriculture, forestry and fishing",0,"Asian, Asian British or Asian Welsh: Bangladeshi",833,10km to less than 30km,0,2 cars or vans in household,3034
2,0-4,Female,Couple household: All children non-dependent,0,Does not apply,34378,"A Agriculture, forestry and fishing",0,"Asian, Asian British or Asian Welsh: Bangladeshi",833,10km to less than 30km,0,3 or more cars or vans in household,1723
3,0-4,Female,Couple household: All children non-dependent,0,Does not apply,34378,"A Agriculture, forestry and fishing",0,"Asian, Asian British or Asian Welsh: Bangladeshi",833,10km to less than 30km,0,No cars or vans in household,1013
4,0-4,Female,Couple household: All children non-dependent,0,Does not apply,34378,"A Agriculture, forestry and fishing",0,"Asian, Asian British or Asian Welsh: Bangladeshi",833,30km to less than 60km,0,1 car or van in household,3139


(13789440, 14)


### 4. Age-related Constraints

The representative population pool is generated by combining multiple
demographic attributes into representative profiles. As a result,
some age groups may be paired with economic activity categories that
are not logically applicable.

Children aged **0–4** and **5–11** are not considered part of the
economically active population. Therefore, their economic activity
status is assigned as **"Does not apply"** to maintain logical
consistency within the representative population.

This constraint is applied before the employment-related constraints,
allowing employment-dependent attributes (e.g., industry and commuting
distance) to be updated consistently in subsequent processing.

In [4]:
YOUNG_CHILDREN = ["0-4", "5-11"]

AGE_MASK = (
    population_pool["Age"].isin(YOUNG_CHILDREN)
)

print("Economic Activity before applying age-related constraints")

display(
    population_pool.loc[
        AGE_MASK,
        "Economic Activity Status"
    ].value_counts()
)

Economic Activity before applying age-related constraints


Economic Activity Status
Does not apply                                                                                                                                                        437760
Economically active and a full-time student: In employment                                                                                                            437760
Economically inactive (excluding full-time students)                                                                                                                  437760
Economically active and a full-time student: Unemployed: Seeking work or waiting to start a job already obtained: Available to start working within 2 weeks           437760
Economically inactive and a full-time student                                                                                                                         437760
Economically active (excluding full-time students): In employment                                             

### 4.1 Before

In [5]:
age_inconsistent = (
    AGE_MASK &
    (
        population_pool["Economic Activity Status"]
        != "Does not apply"
    )
)

print(
    f"Inconsistent economic activity profiles identified: "
    f"{age_inconsistent.sum():,}"
)

Inconsistent economic activity profiles identified: 2,626,560


### 4.2 Apply Constraint

In [6]:
population_pool = (
    population_pool.loc[~age_inconsistent]
    .copy()
    .reset_index(drop=True)
)

### 4.3 After Applying Constraint

In [7]:
print("Economic Activity after applying age-related constraints")

display(
    population_pool.loc[
        population_pool["Age"].isin(YOUNG_CHILDREN),
        "Economic Activity Status"
    ].value_counts()
)

Economic Activity after applying age-related constraints


Economic Activity Status
Does not apply    437760
Name: count, dtype: int64

In [8]:
display(
    population_pool.loc[
        population_pool["Age"].isin(YOUNG_CHILDREN),
        ["Economic Activity Status", "Economic Count"]
    ]
    .drop_duplicates()
    .sort_values("Economic Activity Status")
)

,Economic Activity Status,Economic Count
0,Does not apply,34378
218880,Does not apply,51349


### 5. Employment-related Constraints

Employment-related attributes are only applicable to individuals who
are classified as being in employment.

For representative profiles that are not in employment, the attributes
**Industry** and **Distance travelled to work** are assigned
**"Does not apply"** to maintain logical consistency within the
representative population.

This constraint ensures that work-related attributes are only retained
for representative profiles with valid employment status.

### 5.1 Before Applying Constraint

In [9]:
EMPLOYED_STATUS = [
    "Economically active (excluding full-time students): In employment",
    "Economically active and a full-time student: In employment"
]

NON_EMPLOYED_MASK = (
    ~population_pool["Economic Activity Status"].isin(
        EMPLOYED_STATUS
    )
)

### 5.2 Industry(Before)

In [10]:
print("Industry before applying employment-related constraints")

display(
    population_pool.loc[
        NON_EMPLOYED_MASK,
        "Industry"
    ].value_counts()
)

Industry before applying employment-related constraints


Industry
A Agriculture, forestry and fishing                                        426240
F Construction                                                             426240
O Public administration and defence; compulsory social security            426240
N Administrative and support service activities                            426240
H Transport and storage                                                    426240
E Water supply; sewerage, waste management and remediation activities      426240
P Education                                                                426240
Does not apply                                                             426240
M Professional, scientific and technical activities                        426240
I Accommodation and food service activities                                426240
D Electricity, gas, steam and air conditioning supply                      426240
Q Human health and social work activities                                  426240
J Infor

### 5.3 Distance(Before)

In [11]:
print("Distance travelled to work before applying employment-related constraints")

display(
    population_pool.loc[
        NON_EMPLOYED_MASK,
        "Distance travelled to work"
    ].value_counts()
)

Distance travelled to work before applying employment-related constraints


Distance travelled to work
10km to less than 30km                                                            1012320
30km to less than 60km                                                            1012320
5km to less than 10km                                                             1012320
60km and over                                                                     1012320
Does not apply                                                                    1012320
Less than 5km                                                                     1012320
Works mainly at an offshore installation, in no fixed place, or outside the UK    1012320
Works mainly from home                                                            1012320
Name: count, dtype: int64

### 5.4 Number of inconsistencies identified

In [12]:
industry_inconsistent = (
    NON_EMPLOYED_MASK &
    (
        population_pool["Industry"]
        != "Does not apply"
    )
)

distance_inconsistent = (
    NON_EMPLOYED_MASK &
    (
        population_pool["Distance travelled to work"]
        != "Does not apply"
    )
)

print(
    f"Inconsistent industry profiles identified: "
    f"{industry_inconsistent.sum():,}"
)

print(
    f"Inconsistent commuting distance profiles identified: "
    f"{distance_inconsistent.sum():,}"
)

Inconsistent industry profiles identified: 7,672,320
Inconsistent commuting distance profiles identified: 7,086,240


### 5.5 Apply Constraint

In [13]:
employment_inconsistent = (
    industry_inconsistent
    | distance_inconsistent
)

population_pool = (
    population_pool.loc[~employment_inconsistent]
    .copy()
    .reset_index(drop=True)
)

### 5.6 After Applying Constraint

In [14]:
print("Industry after applying employment-related constraints")

display(
    population_pool.loc[
        NON_EMPLOYED_MASK,
        "Industry"
    ].value_counts()
)

Industry after applying employment-related constraints


Industry
Does not apply                                                             162480
C Manufacturing                                                            122320
A Agriculture, forestry and fishing                                        122080
I Accommodation and food service activities                                121840
J Information and communication                                            121840
N Administrative and support service activities                            121840
H Transport and storage                                                    121840
F Construction                                                             121760
B Mining and quarrying                                                     121760
G Wholesale and retail trade; repair of motor vehicles and motor cycles    121680
L Real estate activities                                                   121600
R, S, T, U Other                                                           121520
Q Human

In [15]:
print("Distance travelled to work after applying employment-related constraints")

display(
    population_pool.loc[
        NON_EMPLOYED_MASK,
        "Distance travelled to work"
    ].value_counts()
)

Distance travelled to work after applying employment-related constraints


Distance travelled to work
Does not apply                                                                    329700
10km to less than 30km                                                            288840
30km to less than 60km                                                            288840
5km to less than 10km                                                             288840
60km and over                                                                     288840
Less than 5km                                                                     288820
Works mainly at an offshore installation, in no fixed place, or outside the UK    288820
Works mainly from home                                                            288820
Name: count, dtype: int64

## 6. Household-related Constraints
Household-related constraints are applied using the semantic definitions of the household categories. Unlike employment-related attributes, logically inconsistent household types cannot be replaced with a meaningful "Does not apply" category. Therefore, infeasible Age–Household Type combinations are removed from the representative population.

Three constraints are applied:

1. Household types in which all household members are aged 66 years and over are restricted to the older age groups.
2. Young children cannot be assigned to household types in which all children are non-dependent.
3. Young children cannot be assigned to couple households with no children.

### 6.1 Define household constraints

In [16]:
OLDER_AGE_GROUPS = ["65-74", "75+"]
YOUNG_CHILDREN = ["0-4", "5-11"]

OLDER_HOUSEHOLD = (
    "Older family household"
)

NON_DEPENDENT_CHILD_HOUSEHOLDS = [
    "Couple household: All children non-dependent",
    "Lone parent household: All children non-dependent"
]

NO_CHILDREN_HOUSEHOLDS = [
    "Couple household: No children"
]

In [17]:
household_types = set(
    population_pool["Household Type"].unique()
)

expected_household_types = {
    OLDER_HOUSEHOLD,
    *NON_DEPENDENT_CHILD_HOUSEHOLDS,
    *NO_CHILDREN_HOUSEHOLDS
}

missing_household_types = (
    expected_household_types
    - household_types
)

print(
    "Missing household categories: "
    f"{missing_household_types}"
)

Missing household categories: set()


### 6.2 Identify inconsistencies

In [18]:
older_household_inconsistent = (
    population_pool["Household Type"].eq(
        OLDER_HOUSEHOLD
    )
    & ~population_pool["Age"].isin(
        OLDER_AGE_GROUPS
    )
)

non_dependent_child_inconsistent = (
    population_pool["Household Type"].isin(
        NON_DEPENDENT_CHILD_HOUSEHOLDS
    )
    & population_pool["Age"].isin(
        YOUNG_CHILDREN
    )
)

no_children_inconsistent = (
    population_pool["Household Type"].isin(
        NO_CHILDREN_HOUSEHOLDS
    )
    & population_pool["Age"].isin(
        YOUNG_CHILDREN
    )
)

household_inconsistent = (
    older_household_inconsistent
    | non_dependent_child_inconsistent
    | no_children_inconsistent
)

In [19]:
print(
    "Inconsistent older-family profiles: "
    f"{older_household_inconsistent.sum():,}"
)

print(
    "Young children in non-dependent-child households: "
    f"{non_dependent_child_inconsistent.sum():,}"
)

print(
    "Young children in no-children households: "
    f"{no_children_inconsistent.sum():,}"
)

print(
    "Total household-inconsistent profiles: "
    f"{household_inconsistent.sum():,}"
)

Inconsistent older-family profiles: 247,520
Young children in non-dependent-child households: 640
Young children in no-children households: 320
Total household-inconsistent profiles: 248,480


### 6.3 Apply constraints

In [20]:
profiles_before_household_constraints = len(
    population_pool
)

population_pool = (
    population_pool.loc[
        ~household_inconsistent
    ]
    .copy()
    .reset_index(drop=True)
)

profiles_after_household_constraints = len(
    population_pool
)

print(
    "Profiles before household constraints: "
    f"{profiles_before_household_constraints:,}"
)

print(
    "Profiles after household constraints: "
    f"{profiles_after_household_constraints:,}"
)

print(
    "Profiles removed: "
    f"{profiles_before_household_constraints - profiles_after_household_constraints:,}"
)

Profiles before household constraints: 3,117,600
Profiles after household constraints: 2,869,120
Profiles removed: 248,480


### 6.4 Validate household constraints

In [21]:
remaining_older_household_inconsistencies = (
    population_pool["Household Type"].eq(
        OLDER_HOUSEHOLD
    )
    & ~population_pool["Age"].isin(
        OLDER_AGE_GROUPS
    )
)

remaining_non_dependent_inconsistencies = (
    population_pool["Household Type"].isin(
        NON_DEPENDENT_CHILD_HOUSEHOLDS
    )
    & population_pool["Age"].isin(
        YOUNG_CHILDREN
    )
)

remaining_no_children_inconsistencies = (
    population_pool["Household Type"].isin(
        NO_CHILDREN_HOUSEHOLDS
    )
    & population_pool["Age"].isin(
        YOUNG_CHILDREN
    )
)

assert not remaining_older_household_inconsistencies.any()
assert not remaining_non_dependent_inconsistencies.any()
assert not remaining_no_children_inconsistencies.any()

print("All household-related constraints passed.")

All household-related constraints passed.


## 7. Final Consistency Checks

In [22]:
young_children = population_pool["Age"].isin(
    ["0-4", "5-11"]
)

young_children_economic_inconsistent = (
    young_children
    & population_pool["Economic Activity Status"].ne(
        "Does not apply"
    )
)

non_employed = (
    ~population_pool["Economic Activity Status"].isin(
        EMPLOYED_STATUS
    )
)

non_employed_industry_inconsistent = (
    non_employed
    & population_pool["Industry"].ne(
        "Does not apply"
    )
)

non_employed_distance_inconsistent = (
    non_employed
    & population_pool["Distance travelled to work"].ne(
        "Does not apply"
    )
)

assert not young_children_economic_inconsistent.any()
assert not non_employed_industry_inconsistent.any()
assert not non_employed_distance_inconsistent.any()
assert not remaining_older_household_inconsistencies.any()
assert not remaining_non_dependent_inconsistencies.any()
assert not remaining_no_children_inconsistencies.any()

print("All representative population constraints passed.")
print(f"Final population shape: {population_pool.shape}")

All representative population constraints passed.
Final population shape: (2869120, 14)


### 8. Export Representative Population

In [23]:
representative_population_file = (
    output_path
    / "representative_population.parquet"
)

population_pool.to_parquet(
    representative_population_file,
    index=False
)

print(
    "Representative population exported to:"
    f"\n{representative_population_file}"
)

print(
    f"Final population shape: {population_pool.shape}"
)

Representative population exported to:
c:\Users\user\OneDrive - University of Stirling\Desktop\dissertation\mobility-modelling-framework\output\demographic_profiles\representative_population.parquet
Final population shape: (2869120, 14)


In [24]:
attribute_columns = [
    "Age",
    "Gender",
    "Household Type",
    "Economic Activity Status",
    "Industry",
    "Ethnic Group",
    "Distance travelled to work",
    "Car Ownership"
]

duplicate_profiles = population_pool.duplicated(
    subset=attribute_columns
).sum()

print(
    f"Duplicate representative profiles: "
    f"{duplicate_profiles:,}"
)

Duplicate representative profiles: 0


In [25]:
full_row_duplicates = population_pool.duplicated().sum()

print(
    f"Completely identical rows: "
    f"{full_row_duplicates:,}"
)

Completely identical rows: 0
